# 05 Performance Analysis

Author: Endri Sojeva

## Reproduce the run

Use the project environment. The Kaggle edition loads the package, benchmark code and fixtures from a private snapshot pinned to a full GitHub SHA. CSV data, metadata and PNG figures are saved under the run's `results/` directory.

In [ ]:
import json
import os
import sys
from pathlib import Path

from IPython.display import Image, display
if (Path.cwd().parent / "experiments/benchmark.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))
from experiments import benchmark

root = Path(benchmark.__file__).resolve().parents[1]
results_root = globals().get("RESULTS_DIR", root / "results")
output = results_root / "notebook-05"
profile = os.environ.get("TIMETRAVEL_BENCHMARK_PROFILE", "full")
raw, summary = benchmark.run_benchmark(
    output, profile=profile, repetitions=3, source_sha=globals().get("SOURCE_COMMIT"),
)
metadata = json.loads((output / "environment.json").read_text(encoding="utf-8"))
verified = raw[raw["suite"] == "detection"]
tradeoff = summary[summary["scenario"] == "repair_tradeoff"].set_index("algorithm")
assert verified["truth_match"].eq(True).all()
assert tradeoff.loc["greedy", "cost_median"] == 6
assert tradeoff.loc["branch_and_bound", "cost_median"] == 5
assert tradeoff.loc["branch_and_bound", "proven_trials"] == 3
print(json.dumps({"source_sha": metadata["source_sha"], "profile": profile, "trials": len(raw),
                  "verified_detections": len(verified), "max_events": int(raw["events"].max()),
                  "greedy_cost": 6, "proven_cost": 5, "libraries": metadata["libraries"]}))

## Experimental design

The full profile uses DAGs with 10, 100, 500 and 2,000 events, two forward-edge probabilities and two fixed seeds. Named travel loops, mixed conflicts and random backward edges exercise separate behaviors. A fixed-size repair sweep varies the number of contradictory assertion groups and the number of eligible edits. Compare timings within the same workload and environment.

In [ ]:
scaling = summary[summary["family"] == "dag"].copy()
for stat in ("q1", "median", "q3"):
    scaling[f"ms_{stat}"] = scaling[f"runtime_seconds_{stat}"] * 1000
scaling["peak_KiB"] = scaling["peak_python_bytes_median"] / 1024
display(scaling[["algorithm", "events", "relations", "edge_density", "trials", "ms_q1", "ms_median", "ms_q3", "peak_KiB"]])
display(Image(filename=str(output / "figures" / "detection_scaling.png")))

## Candidates and pruning

A candidate is one attempted edit. A prefix is expanded when it is inconsistent and the search enters its edit loop. Exact prefix discard share is the number of cost-bound and dominance cuts divided by those cuts plus exact expansions. For catalogs up to eight edits, raw-space pruning counts descendants skipped below explicit nonterminal cuts, divided by all nonempty ordered edit prefixes. These shares have different denominators. Short-circuit consistency, invalid edits and time or candidate limits do not count as explicit raw-prefix cuts.

In [ ]:
sweep = summary[summary["family"] == "conflict-sweep"]
display(sweep[["scenario", "algorithm", "catalog_entries", "contradictions", "candidates_explored_median",
               "exact_prefix_discard_share_pct_median", "raw_prefix_space_pruned_pct_median", "statuses"]])
display(Image(filename=str(output / "figures" / "repair_search.png")))
print(metadata)

## Interpretation and limitations

DFS and Tarjan are O(V+E), but elapsed time also depends on early exit, graph density and allocation behavior. An observed trend on this grid is not an empirical proof of complexity. Repair may grow much faster because the number of ordered prefixes grows factorially in the catalog size. More conflicts in this sweep also introduce more candidate edits, so their effects are not isolated. Time is measured with `perf_counter` and peak Python allocations with a separate `tracemalloc` replay. The latter excludes untracked native allocations and process RAM. The raw CSV records the replay status and candidate count for limit-sensitive repair runs.